**Cell 1**

In [1]:
import os

print("📂 EVERYTHING INSIDE /kaggle/input/:")
try:
    for item in os.listdir("/kaggle/input/"):
        print(f" ├── {item}")
        
        # Let's peek one level deeper to be safe
        item_path = os.path.join("/kaggle/input/", item)
        if os.path.isdir(item_path):
            for sub_item in os.listdir(item_path)[:3]: # Just print first 3 items
                print(f" │    ├── {sub_item}")
            print(" │    └── ...")
except Exception as e:
    print("Error:", e)

📂 EVERYTHING INSIDE /kaggle/input/:
 ├── datasets
 │    ├── jeromeblanchet
 │    ├── muhammaddanish100
 │    └── ...


In [2]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121

# Install Unsloth
!pip install "unsloth==2025.3.19"
!pip install transformers==4.51.3 trl==0.16.1 accelerate==1.6.0 peft huggingface_hub
!pip uninstall cut-cross-entropy -y

# Remove incompatible library
!pip uninstall torchao -y

# **Cell 2**

In [3]:
import json
import sqlite3
import os

def schema_dict_to_create_sql(schema_dict):
    """
    Parses Spider 'tables.json' to generate accurate SQL DDL.
    Improvements:
    1. Uses ACTUAL column types (number -> INTEGER, text -> TEXT).
    2. Adds FOREIGN KEY constraints (Critical for JOINs).
    """
    try:
        # standard spider keys
        table_names = schema_dict.get("table_names_original", [])
        column_names = schema_dict.get("column_names_original", []) # [[table_idx, "col_name"], ...]
        column_types = schema_dict.get("column_types", []) # ["text", "number", ...]
        primary_keys = schema_dict.get("primary_keys", []) # [col_idx, ...]
        foreign_keys = schema_dict.get("foreign_keys", []) # [[src_col_idx, tgt_col_idx], ...]

        # Fallback for non-standard schemas
        if not table_names: 
            return str(schema_dict)

        # 1. Map Table IDs to Names
        table_map = {i: name for i, name in enumerate(table_names)}
        
        # 2. Organize Columns by Table
        # Structure: {table_id: [(col_idx, col_name, col_type), ...]}
        table_cols = {}
        for c_idx, (t_id, c_name) in enumerate(column_names):
            if t_id < 0: continue # Skip special '*' columns
            dtype = column_types[c_idx] if c_idx < len(column_types) else "text"
            table_cols.setdefault(t_id, []).append((c_idx, c_name, dtype))

        # 3. Build CREATE Statements
        create_stmts = []
        
        for t_id, cols in table_cols.items():
            t_name = table_map.get(t_id, f"table_{t_id}")
            definitions = []
            
            # -- Columns --
            for c_idx, c_name, c_type in cols:
                # Map Spider types to SQL types
                sql_type = "INTEGER" if c_type == "number" else "TEXT"
                def_str = f'"{c_name}" {sql_type}'
                
                # Add Primary Key inline (simplest for LLMs)
                if c_idx in primary_keys:
                    def_str += " PRIMARY KEY"
                
                definitions.append(def_str)
            
            # -- Foreign Keys --
            # Find FKs where the source column belongs to THIS table
            for src_idx, tgt_idx in foreign_keys:
                # check if src_idx is in the current table's columns
                if src_idx in [c[0] for c in cols]:
                    # Get target details
                    tgt_t_id = column_names[tgt_idx][0]
                    tgt_t_name = table_map.get(tgt_t_id, "unknown")
                    tgt_c_name = column_names[tgt_idx][1]
                    
                    # Get source column name
                    src_c_name = column_names[src_idx][1]
                    
                    fk_str = f'FOREIGN KEY ("{src_c_name}") REFERENCES {tgt_t_name}("{tgt_c_name}")'
                    definitions.append(fk_str)

            stmt = f"CREATE TABLE {t_name} (\n  " + ",\n  ".join(definitions) + "\n);"
            create_stmts.append(stmt)

        return "\n\n".join(create_stmts)

    except Exception as e:
        print(f"Schema Parse Error: {e}")
        return ""

def execute_sql(sql, db_path):
    """Executes SQL and returns results (list of tuples) or Error string."""
    if not sql: return "Error: Empty SQL"
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(sql)
        result = cursor.fetchall()
        conn.close()
        return result
    except Exception as e:
        return f"Error: {e}"

# Cell 3

In [4]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
import json
import os

# --- NUCLEAR FIX (Prevent Inference Crashes) ---
def no_op_compile(model, *args, **kwargs): return model
torch.compile = no_op_compile
torch._dynamo.config.disable = True
# -----------------------------------------------

# --- CONFIG ---
# ✅ POINTING TO YOUR LLAMA 3.1 CHECKPOINT
# --- 🛑 CRITICAL DIRECTORY SETUP 🛑 ---

# Your Model (Nested under your username!)
CHECKPOINT_PATH = "/kaggle/input/datasets/muhammaddanish100/llama-3-1-nl2sql-finetuned/outputs/checkpoint-881"

# The Spider Benchmark
SPIDER_PATH = "/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider"

# --------------------------------------

print(f"📂 Loading Fine-Tuned Adapter from: {CHECKPOINT_PATH}")

# 1. Load the Model + Adapters
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = CHECKPOINT_PATH, 
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

# 2. Enable Inference Mode (2x Faster)
FastLanguageModel.for_inference(model)
tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")  # ← LLAMA 3.1

# 3. Load Spider Data (Validation Set)
with open(f"{SPIDER_PATH}/dev.json", 'r') as f:
    dev_data = json.load(f)

# 4. Load & Parse Schemas
with open(f"{SPIDER_PATH}/tables.json", 'r') as f:
    tables_data = json.load(f)
    schema_lookup = {t['db_id']: schema_dict_to_create_sql(t) for t in tables_data}

print(f"✅ Successfully loaded model and {len(dev_data)} test examples.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-04-26 02:55:03.276915: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777172103.445654      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777172103.496234      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777172103.897931      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777172103.897969      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777172103.897971      23 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


📂 Loading Fine-Tuned Adapter from: /kaggle/input/datasets/muhammaddanish100/llama-3-1-nl2sql-finetuned/outputs/checkpoint-881
==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:30: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Unsloth 2025.3.19 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Successfully loaded model and 1034 test examples.


# Cell 4

In [5]:
import re
from tqdm import tqdm

def normalize_sql(sql):
    if not sql: return ""
    sql = sql.lower().strip().rstrip(";")
    sql = re.sub(r'\s+', ' ', sql)
    return sql

def is_set_match(gold_res, pred_res):
    if isinstance(gold_res, str) or isinstance(pred_res, str): return False
    if not gold_res or not pred_res: return False
    try:
        return set(gold_res) == set(pred_res)
    except:
        return False

# --- CONFIG ---
NUM_SAMPLES = len(dev_data) # Run on ALL examples
results = []

print(f"🚀 Starting Full Evaluation on {NUM_SAMPLES} samples...")

for i in tqdm(range(NUM_SAMPLES)):
    item = dev_data[i]
    db_id = item['db_id']
    question = item['question']
    gold_sql = item['query']
    
    # 1. Get Schema
    schema = schema_lookup.get(db_id, "")
    
    # 2. Prompting
    SYSTEM_PROMPT = (
    "You are an expert Text-to-SQL model. Generate a valid SQLite query to answer the user's question, "
    "using ONLY the tables and columns provided in the schema.\n"
    "Follow these strict rules:\n"
    "1. Output ONLY the raw SQL code. Do not use markdown format or explanations.\n"
    "2. Do not invent tables or columns that do not exist in the schema.\n"
    "3. Return columns in the EXACT order requested in the question.\n"
    "4. Do NOT join tables unless strictly necessary to connect data. Avoid redundant joins.\n"
    "5. Prefer 'ORDER BY ... LIMIT 1' over subqueries for finding maximums or top results.\n"
    "6. Apply all categorical filters (WHERE clauses) from the question before applying ordering or limits."
    )
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Schema:\n{schema}\n\nQuestion:\n{question}"},
    ]
    
    # 3. Generate
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        input_ids=inputs, 
        max_new_tokens=150, 
        use_cache=True, 
        temperature=0.0, 
        do_sample=False
    )
    pred_raw = tokenizer.batch_decode(outputs)[0]
    
    # 4. Extract SQL (LLAMA 3.1 FORMAT)
    try:
        pred_sql = pred_raw.split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip()
        pred_sql = pred_sql.replace("```sql", "").replace("```", "").strip()
        if "\n" in pred_sql and not pred_sql.upper().startswith("SELECT"):
             lines = pred_sql.split('\n')
             pred_sql = lines[-1] 
    except:
        pred_sql = "ERROR_PARSING"

    # ============================================================
    # 🛠️ TOKENIZATION FIX
    # This repairs the split operators like "> =" into ">="
    # ============================================================
    pred_sql = re.sub(r'>\s+=', '>=', pred_sql)
    pred_sql = re.sub(r'<\s+=', '<=', pred_sql)
    pred_sql = re.sub(r'!\s+=', '!=', pred_sql)
    # ============================================================

    # 5. Execute & Compare
    db_file = f"{SPIDER_PATH}/database/{db_id}/{db_id}.sqlite"
    gold_res = execute_sql(gold_sql, db_file)
    pred_res = execute_sql(pred_sql, db_file)
    
    # Metrics
    ex_match = (str(gold_res) == str(pred_res)) and ("Error" not in str(pred_res))
    
    soft_match = False
    if not ex_match and "Error" not in str(pred_res):
        soft_match = is_set_match(gold_res, pred_res)
    
    # Save Result
    results.append({
        "db_id": db_id,
        "question": question,
        "gold": gold_sql,
        "pred": pred_sql,
        "ex_match": ex_match,
        "soft_match": soft_match
    })

# --- FINAL REPORT ---
strict_acc = sum(1 for r in results if r['ex_match']) / len(results)
relaxed_acc = sum(1 for r in results if r['ex_match'] or r['soft_match']) / len(results)

print(f"\n📊 FINAL RESULTS ({len(results)} samples):")
print(f"✅ Strict Accuracy:  {strict_acc:.2%}")
print(f"🤝 Relaxed Accuracy: {relaxed_acc:.2%}")

🚀 Starting Full Evaluation on 1034 samples...


100%|██████████| 1034/1034 [2:23:07<00:00,  8.30s/it]


📊 FINAL RESULTS (1034 samples):
✅ Strict Accuracy:  67.79%
🤝 Relaxed Accuracy: 70.31%


# Cell 5

In [6]:
import pandas as pd

# Filter: Show ONLY cases where the data returned was wrong (Soft Match Failed)
# This ignores simple ordering differences and focuses on logic errors.
real_failures = [r for r in results if not r['soft_match'] and not r['ex_match']]

print(f"❌ Found {len(real_failures)} Logic/Syntax Failures (ignoring ordering differences).\n")
print("="*60)

# Loop through top failures
for i, fail in enumerate(real_failures[:10]): # Show top 10
    print(f"🔴 FAILURE CASE #{i+1} (DB: {fail['db_id']})")
    
    print(f"❓ Q: {fail['question']}")
    print(f"✅ GOLD: {fail['gold']}")
    print(f"❌ PRED: {fail['pred']}")
    
    print("="*60)
    print("\n")

❌ Found 307 Logic/Syntax Failures (ignoring ordering differences).

🔴 FAILURE CASE #1 (DB: concert_singer)
❓ Q: Show the name and the release year of the song by the youngest singer.
✅ GOLD: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
❌ PRED: SELECT T1.Name,  T1.Song_Name FROM singer AS T1 JOIN singer_in_concert AS T2 ON T1.Singer_ID  =  T2.Singer_ID JOIN concert AS T3 ON T2.concert_ID  =  T3.concert_ID WHERE T1.Age  =  (SELECT MIN(Age) FROM singer)


🔴 FAILURE CASE #2 (DB: concert_singer)
❓ Q: What are the names and release years for all the songs of the youngest singer?
✅ GOLD: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
❌ PRED: SELECT T1.Name,  T1.Song_Name,  T1.Song_release_year FROM singer AS T1 JOIN singer_in_concert AS T2 ON T1.Singer_ID  =  T2.Singer_ID JOIN concert AS T3 ON T2.concert_ID  =  T3.concert_ID WHERE T1.Age  =  (SELECT MIN(Age) FROM singer)


🔴 FAILURE CASE #3 (DB: concert_singer)
❓ Q: List all song names by sing